In [ ]:
import torch.nn as nn
from torch.amp import autocast, GradScaler
from torch.utils.data import Dataset, DataLoader, random_split
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
import json
import matplotlib.pyplot as plt
from tqdm import tqdm
import evaluate
import math

import torch, gc
gc.collect()
torch.cuda.empty_cache()

LEARNING_RATE = 5e-5
REPETITION_PENALTY = 1.2

# Assuming the model download finished successfully
MODEL_LOCAL_DIR = MODEL_DIR / qwen3.5

# Pass the LOCAL folder path, not the model name
tokenizer = AutoTokenizer.from_pretrained(MODEL_LOCAL_DIR)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",           # High-accuracy 4-bit type
    bnb_4bit_use_double_quant=True,      # Quantize the quantizers for extra space
    bnb_4bit_compute_dtype=torch.float16 # Math still happens in 16-bit
)

base_model = AutoModelForCausalLM.from_pretrained(local_folder,
                                                #   quantization_config=bnb_config,
                                                  device_map="auto")
base_model.config.use_cache = False
base_model.gradient_checkpointing_enable()
print(f"Base model: {base_model}")

geometric_tokens = [
    # --- Control & Formatting Tokens ---
    # "<GEO_START>", "<GEO_END>", "<DESCRIBE>", "<PAD>",
    "<GEO_START>", "<GEO_END>", "<DESCRIBE>",

    # --- ISO 10303 Topology Tokens (The Graph) ---
    "<VERTEX_POINT>",
    "<EDGE_CURVE>", "<ORIENTED_EDGE>",
    "<EDGE_LOOP>",
    "<FACE_BOUND>", "<FACE_OUTER_BOUND>", "<ADVANCED_FACE>",
    "<CLOSED_SHELL>", "<OPEN_SHELL>",
    "<MANIFOLD_SOLID_BREP>",

    # --- ISO 10303 Surface Geometry Tokens (The 2D Math) ---
    "<PLANE>",
    "<CYLINDRICAL_SURFACE>", "<CONICAL_SURFACE>",
    "<SPHERICAL_SURFACE>", "<TOROIDAL_SURFACE>",
    "<SURFACE_OF_LINEAR_EXTRUSION>", "<SURFACE_OF_REVOLUTION>",
    "<B_SPLINE_SURFACE_WITH_KNOTS>", "<RATIONAL_B_SPLINE_SURFACE>",

    # --- ISO 10303 Curve Geometry Tokens (The 1D Math) ---
    "<LINE>", "<CIRCLE>", "<ELLIPSE>",
    "<PARABOLA>", "<HYPERBOLA>",
    "<B_SPLINE_CURVE_WITH_KNOTS>", "<RATIONAL_B_SPLINE_CURVE>",

    # --- ISO 10303 Primitive Math Tokens ---
    "<CARTESIAN_POINT>", "<DIRECTION>", "<VECTOR>", "<AXIS2_PLACEMENT_3D>",

    # --- Macro compressed tokens ---
    "<CYLINDER_PRIMITIVE>", # A CYLINDRICAL_SURFACE node connected to exactly two CIRCLE edge nodes
    "<FILLET_CHAIN>", # Two or more B_SPLINE_SURFACE patches that share a common boundary edge and exhibit C^1 (tangential) continuity
    "<THROUGH_HOLE>", # A CYLINDRICAL_SURFACE where its two bounding CIRCLE edges are each connected to separate, distinct PLANE surfaces with opposing normal vectors.
    "<CHAMFER_EDGE>", #A CONICAL_SURFACE or narrow PLANE that bridges two orthogonal PLANE surfaces, bounded by parallel LINE or CIRCLE edges.
    "<SPHERE_PRIMITIVE>", # A SPHERICAL_SURFACE bounded by a single CIRCLE edge (forming a hemisphere or ball joint) or a single vertex pole.
    "<PLANAR_PAD>", # A flat PLANE bounded by an EDGE_LOOP, where every edge in the loop connects perpendicularly to a surrounding "skirt" of PLANE or CYLINDRICAL_SURFACE nodes.

    # Marker for different annotation levels.
    "[L1]", "[L2]", "[L3]",

    # Marker for output.
    "<ANNOTATE>",
]

num_added_toks = tokenizer.add_special_tokens({'additional_special_tokens': geometric_tokens})

# Ensure there is a padding token
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({'pad_token': '<PAD>'})
    num_added_toks += 1

print(f"Added {num_added_toks} new tokens. New vocab size: {len(tokenizer)} tokens")

# Resize the base model's embedding matrix to accommodate the new tokens
base_model.resize_token_embeddings(len(tokenizer))

# ---------------------------------------------------------
# 1. Dataset Definition
# ---------------------------------------------------------
class CADGeometricDataset(Dataset):
    def __init__(self, paths, tokenizer, max_length=1024):
        self.data = []
        for path in paths:
            with open(path, "r") as f:
                for line in f:
                    line = line.strip()
                    if line:
                        self.data.append(json.loads(line))

        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]

        # Concatenate the semantic geometry tokens with the Ground Truth annotation
        # The model learns to predict the hierarchical target based on the geometry
        full_text = "".join(item["token_stream"]) + "<ANNOTATE>" + item["annotation"] + "<|endoftext|>"

        encoding = self.tokenizer(
            full_text,
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt"
        )

        # Load Stream B (vectors)
        vectors = torch.tensor(item["vector_stream"], dtype=torch.float32)
        padded_vectors = torch.zeros((self.max_length, 128))

        seq_len = min(len(vectors), self.max_length)
        padded_vectors[:seq_len, :] = vectors[:seq_len, :]

        input_ids = encoding["input_ids"].squeeze()
        labels = input_ids.clone()

        # Mask the padding tokens
        labels[input_ids == self.tokenizer.pad_token_id] = -100

        # Mask the Prompt (Everything up to and including <ANNOTATE>)
        annotate_id = self.tokenizer.convert_tokens_to_ids("<ANNOTATE>")
        matches = (input_ids == annotate_id).nonzero(as_tuple=True)[0]

        if len(matches) > 0:
            anno_idx = matches[0].item()
            labels[:anno_idx + 1] = -100 # -100 masks the padding token so the model doesn't train on them.
        else:
            print(f"WARNING: File {item['file_path']} is too big.")
            # Handle large files by preventing training
            labels[:] = -100

            # Keep one single valid token at the end so the CrossEntropyLoss
            # doesn't try to divide by zero and return a NaN loss.
            labels[-1] = self.tokenizer.eos_token_id


        return {
            "input_ids": input_ids,
            "attention_mask": encoding["attention_mask"].squeeze(),
            "geometric_vectors": padded_vectors,
            "labels": labels # Causal LM labeling
        }

# ---------------------------------------------------------
# 2. Dual-Stream Model Architecture
# ---------------------------------------------------------
class GeometricSemanticModel(nn.Module):
    def __init__(self, base_model):
        super().__init__()
        self.base_model = base_model
        # W_p: Projects 128-dim CAD math into 768-dim GPT-2 embedding space
        self.W_p = nn.Linear(128, 768)

    def forward(self, input_ids, geometric_vectors, attention_mask, labels=None):
        # 1. Get standard discrete token embeddings
        # inputs_embeds = self.base_model.transformer.wte(input_ids)
        inputs_embeds = self.base_model.transformer.get_input_embeddings()(input_ids)

        # 2. Project Stream B and fuse with Stream A
        geometric_vectors = geometric_vectors.to(torch.float32)

        # Ensure W_p operates in FP32 to prevent overflow from raw CAD coordinates
        with torch.autocast("cuda", enabled=False):
            geometric_embeds = self.W_p(geometric_vectors)

        # 3. Cast back to match GPT-2's embedding type (FP16 or BF16)
        geometric_embeds = geometric_embeds.to(inputs_embeds.dtype)

        fused_embeds = inputs_embeds + geometric_embeds

        # 3. Pass through the Transformer
        outputs = self.base_model(
            inputs_embeds=fused_embeds,
            attention_mask=attention_mask,
            labels=labels
        )
        return outputs

def annotate(model, tokenizer, start_tokens, prompt_vectors, max_new_tokens=50, device="cuda"):
    model.eval()
    generated_ids = start_tokens.clone()

    # Start with the geometric vectors that match the prompt
    current_vectors = prompt_vectors.clone()

    with torch.no_grad():
        for _ in range(max_new_tokens):
            if generated_ids.shape[1] >= 1024:
                break

            attn_mask = torch.ones_like(generated_ids).to(device)

            outputs = model(
                input_ids=generated_ids,
                geometric_vectors=current_vectors,
                attention_mask=attn_mask
            )

            # Get the logits for the last predicted token
            next_token_logits = outputs.logits[:, -1, :]

            repetition_penalty = REPETITION_PENALTY
            for prev_id in set(generated_ids[0].tolist()):
                if next_token_logits[0, prev_id] < 0:
                    next_token_logits[0, prev_id] *= repetition_penalty
                else:
                    next_token_logits[0, prev_id] /= repetition_penalty

            next_token_id = torch.argmax(next_token_logits, dim=-1).unsqueeze(-1)

            # 1. Grow the Text Stream
            generated_ids = torch.cat([generated_ids, next_token_id], dim=-1)

            zero_vec = torch.zeros((1, 1, 128)).to(device)
            current_vectors = torch.cat([current_vectors, zero_vec], dim=1)

            # Stop early if the model predicts <|endoftext|>
            if next_token_id.item() == tokenizer.eos_token_id:
                break

    return generated_ids

---------------------------------------------------------
3. Training Loop & Visualization
---------------------------------------------------------
history = {"train_loss": [], "val_loss": [], "val_bertscore": [], "val_rougeL": [], "val_bleu": []}
def train():
    global history
    global base_model
    global tokenizer
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Initializing training on {device}...")

    if isinstance(base_model, PeftModel) or hasattr(base_model, "peft_config"):
        print("Previous PEFT configuration detected. Unloading old adapters...")
        base_model = base_model.unload()

    lora_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        fan_in_fan_out=True,
        # r=8,
        r=16,
        lora_alpha=32,
        lora_dropout=0.1,
        # target_modules=["c_attn"]
        target_modules=["c_attn", "c_proj", "c_fc"] # Targeting GPT-2 attention blocks
    )
    base_model = get_peft_model(base_model, lora_config)
    # Wrap in Dual-Stream Architecture
    model = GeometricSemanticModel(base_model).to(device)

    # Dataloader
    dataset = CADGeometricDataset(annotated_dataset_filepaths, tokenizer)

    metric_bertscore = evaluate.load("bertscore")
    metric_rouge = evaluate.load("rouge")
    metric_bleu = evaluate.load("bleu")

    train_size = int(0.9 * len(dataset))
    val_size = len(dataset) - train_size
    train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

    train_dataloader = DataLoader(train_dataset, batch_size=1, shuffle=True)
    # Validation batch size 1 to process generations sequentially without heavy padding
    val_dataloader = DataLoader(val_dataset, batch_size=1, shuffle=False)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

    epochs = 100

    history = {"train_loss": [], "val_loss":[], "val_bertscore": [], "val_rougeL": [], "val_bleu": []}

    print("Beginning Dual-Stream Fine-Tuning...")

    annotation_token_id = tokenizer.convert_tokens_to_ids("<ANNOTATE>")

    # Set up Gradient Accumulation to handle low GPU memory
    accumulation_steps = 4

    # Initialize the Mixed Precision Scaler
    scaler = GradScaler("cuda")

    for epoch in range(epochs):
        model.train()

        total_loss = 0
        progress_bar = tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{epochs} [Train]")
        optimizer.zero_grad()

        for i, batch in enumerate(progress_bar):
            # Wrap the forward pass in Mixed Precision
            with autocast("cuda", dtype=torch.float16):
                outputs = model(
                    input_ids=batch["input_ids"].to(device),
                    geometric_vectors=batch["geometric_vectors"].to(device),
                    attention_mask=batch["attention_mask"].to(device),
                    labels=batch["labels"].to(device)
                )
                # Scale the loss to account for accumulation
                loss = outputs.loss / accumulation_steps
                if math.isnan(loss.item()):
                    print(f"outputs.loss: {outputs.loss}")

            # Backward pass with scaled gradients
            scaled_loss = scaler.scale(loss)

            # print(f"\n--- SCALING DEBUG ---")
            # if i == 0:
            #     print(f"Scale Factor: {scaler.get_scale()}")
            # print(f"True Loss:    {loss.item():.4f}")
            # print(f"Scaled Loss:  {scaled_loss.item():.4f}")
            scaled_loss.backward()

            # Only step the optimizer every batchs
            if (i + 1) % accumulation_steps == 0 or (i + 1) == len(train_dataloader):

                grad_before = model.W_p.weight.grad.abs().max().item()
                # print(f"Max Gradient (Scaled):   {grad_before:.6f}")

                # 1. Unscale gradients
                scaler.unscale_(optimizer)

                # 2. Clip unscaled gradients at 1.0
                # Because the model is using the AdamW optimizer. AdamW calculates a moving average of past gradients (momentum) and their variances. A gradient spike will cause issues that might lead to NaN / Inf loss.
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

                grad_after = model.W_p.weight.grad.abs().max().item()
                # print(f"Max Gradient (Unscaled): {grad_after:.6f}")
                # print(f"---------------------\n")

                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()

            total_loss += (loss.item() * accumulation_steps)

            progress_bar.set_postfix({"loss": f"{(loss.item() * accumulation_steps):.4f}"})

        avg_train_loss = total_loss / len(train_dataloader)
        history["train_loss"].append(avg_train_loss)
        total_val_loss = 0

        with torch.no_grad():
            for batch in val_dataloader:
                with autocast("cuda", dtype=torch.float16):
                    outputs = model(
                        input_ids=batch["input_ids"].to(device),
                        geometric_vectors=batch["geometric_vectors"].to(device),
                        attention_mask=batch["attention_mask"].to(device),
                        labels=batch["labels"].to(device)
                    )
                    total_val_loss += outputs.loss.item()

        avg_val_loss = total_val_loss / len(val_dataloader)
        history["val_loss"].append(avg_val_loss)

        # --- VALIDATION PHASE (Generative Metrics) ---
        val_preds = []
        val_refs = []

        # Only validate on a subset (e.g., first 20 items) to save time during epochs
        val_subset_limit = 20

        # print(f"Running validation metrics on {val_subset_limit} samples...")
        for i, batch in enumerate(val_dataloader):
            if i >= val_subset_limit: break

            input_ids = batch["input_ids"].to(device)
            geom_vectors = batch["geometric_vectors"].to(device)

            # Find where <ANNOTATION> occurs to split the prompt from the target
            anno_idx = (input_ids[0] == annotation_token_id).nonzero(as_tuple=True)[0]
            if len(anno_idx) == 0:
                continue

            prompt_len = anno_idx[0].item() + 1
            prompt_geom_vectors = geom_vectors[:, :prompt_len, :]
            # print(f"prompt_geom_vectors: {prompt_geom_vectors}")

            prompt_ids = input_ids[:, :prompt_len]

            # Generate the prediction
            generated_ids = annotate(
                model, tokenizer, prompt_ids, prompt_geom_vectors, max_new_tokens=60, device=device
            )

            # Isolate just the newly generated text (ignore the topological prompt)
            new_tokens = generated_ids[0][prompt_len:]
            pred_text = tokenizer.decode(new_tokens, skip_special_tokens=True)

            # Isolate the ground truth reference
            ref_tokens = input_ids[0][prompt_len:]
            # Filter out padding
            ref_tokens = ref_tokens[ref_tokens != tokenizer.pad_token_id]
            ref_text = tokenizer.decode(ref_tokens, skip_special_tokens=True)
            # print(f"ref_text: {ref_text}")
            val_preds.append(pred_text)
            val_refs.append(ref_text)

        print("-------------------------------------------------------------------")
        # Compute Metrics
        safe_val_preds = [p if p.strip() else "EMPTY_PREDICTION" for p in val_preds]
        print(safe_val_preds)
        print(val_refs)
        print("-------------------------------------------------------------------")

        results_bert = metric_bertscore.compute(
            predictions=safe_val_preds,
            references=val_refs,
            lang="en",
            model_type="roberta-base",
            device="cpu"
        )

        results_rouge = metric_rouge.compute(predictions=val_preds, references=val_refs)

        # BLEU requires references to be a list of lists
        bleu_refs = [[r] for r in val_refs]
        results_bleu = metric_bleu.compute(predictions=val_preds, references=bleu_refs)

        avg_bert = sum(results_bert['f1']) / len(results_bert['f1'])

        history["val_bertscore"].append(avg_bert)
        history["val_rougeL"].append(results_rouge['rougeL'])
        history["val_bleu"].append(results_bleu['bleu'])

        print("-------------------------------------------------------------------")
        print(f"Epoch {epoch+1} Results: Training Loss: {avg_train_loss:.4f} | Validation Loss: {avg_val_loss:.4f} | BERTScore: {avg_bert:.4f} | ROUGE-L: {results_rouge['rougeL']:.4f} | BLEU: {results_bleu['bleu']:.4f}")
        print("-------------------------------------------------------------------")